In [2]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from gensim import corpora, models

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ayberkpalta/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ayberkpalta/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ayberkpalta/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ayberkpalta/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
df = pd.read_csv('training.1600000.processed.noemoticon.csv',
                 encoding='latin-1',
                 names=['target', 'ids', 'date', 'flag', 'user', 'text'])

df = df[['text']].sample(5000, random_state=42)
print(df.head(3))

                                                     text
541200             @chrishasboobs AHHH I HOPE YOUR OK!!! 
750     @misstoriblack cool , i have no tweet apps  fo...
766711  @TiannaChaos i know  just family drama. its la...


prep data

In [9]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [w for w in tokens if w.isalpha()]  
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return tokens

df['tokens'] = df['text'].apply(preprocess)
print(df['tokens'].head(3))

541200                      [chrishasboobs, ahhh, hope, ok]
750                [misstoriblack, cool, tweet, apps, razr]
766711    [tiannachaos, know, family, drama, next, time,...
Name: tokens, dtype: object


Create a dictionary and corpus for Gensim LDA

In [10]:
dictionary = corpora.Dictionary(df['tokens'])
corpus = [dictionary.doc2bow(tokens) for tokens in df['tokens']]

print("Sample bag-of-words for first doc:", corpus[0])

Sample bag-of-words for first doc: [(0, 1), (1, 1), (2, 1), (3, 1)]


train LDA

In [11]:
lda_model = models.LdaModel(corpus=corpus,
                            id2word=dictionary,
                            num_topics=5, 
                            passes=10,
                            random_state=42)

topics = lda_model.print_topics(num_words=5)
for topic in topics:
    print(topic)

(0, '0.008*"get" + 0.006*"one" + 0.006*"go" + 0.006*"sad" + 0.005*"home"')
(1, '0.015*"good" + 0.012*"know" + 0.011*"today" + 0.010*"like" + 0.010*"u"')
(2, '0.014*"thanks" + 0.012*"really" + 0.007*"well" + 0.006*"oh" + 0.006*"follow"')
(3, '0.009*"going" + 0.008*"ca" + 0.008*"sleep" + 0.008*"need" + 0.008*"think"')
(4, '0.023*"quot" + 0.016*"day" + 0.013*"na" + 0.010*"good" + 0.008*"http"')


 Get dominant topic for each tweet


In [12]:
def get_topic(doc_bow):
    topics = lda_model.get_document_topics(doc_bow)
    topics = sorted(topics, key=lambda x: -x[1])
    return topics[0][0] if topics else None

df['topic'] = [get_topic(bow) for bow in corpus]
print(df[['text', 'topic']].head(10))

                                                      text  topic
541200              @chrishasboobs AHHH I HOPE YOUR OK!!!       0
750      @misstoriblack cool , i have no tweet apps  fo...      1
766711   @TiannaChaos i know  just family drama. its la...      3
285055   School email won't open  and I have geography ...      2
705995                              upper airways problem       0
379611          Going to miss Pastor's sermon on Faith...       3
1189018            on lunch....dj should come eat with me       2
667030    @piginthepoke oh why are you feeling like that?       2
93541      gahh noo!peyton needs to live!this is horrible       1
1097326  @mrstessyman thank you glad you like it! There...      4
